# CAPS Framework — Full Pipeline Demo

This notebook demonstrates the complete CAPS pipeline on synthetic data
modelled after Nielsen-format retail panel structures.

**Reference:** Inferring Purchasing Power Patterns from Competitive Market Signals:
A Conceptual Framework Integrating Behavioral Dimensionality Reduction and
Qualitative Refinement.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from caps import CAPSPipeline
from caps.acquisition.loaders import load_sample_data, _generate_synthetic_data
from caps.visualization.signal_plots import (
    plot_entropy_weights,
    plot_explained_variance,
    plot_silhouette,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'

## 1. Load Data

In [ ]:
df = _generate_synthetic_data(n_consumers=600, random_state=42)
print(f'Dataset shape: {df.shape}')
print(f'Segments in ground truth: {df.true_segment.value_counts().to_dict()}')
df.head()

## 2. Descriptive Statistics of Signal Dimensions

In [ ]:
signal_cols = [
    'adoption_lag_days', 'brand_switch', 'price_elasticity',
    'attribute_priority_score', 'loyalty_retention_ratio'
]
df[signal_cols].describe().round(3)

## 3. Run CAPS Pipeline

In [ ]:
pipeline = CAPSPipeline(
    observation_window=90,
    variance_threshold=0.72,
    min_silhouette=0.45,
    min_alignment_score=0.70,
    random_state=42,
    max_clusters=8,
)

results = pipeline.fit(df)
results.summary()

## 4. Entropy Weights

In [ ]:
ew = pipeline._weighter
plot_entropy_weights(
    weights=results.entropy_weights,
    entropy=ew.entropy_,
    feature_names=[
        'Adoption Lag', 'Brand Switch', 'Price Elasticity',
        'Attr. Priority', 'Loyalty Retention', 'Category Spend', 'Trial Freq.'
    ][:results.entropy_weights.shape[0]],
)

## 5. Explained Variance (Scree Plot)

In [ ]:
plot_explained_variance(
    evr=results.explained_variance_ratio,
    n_components=results.n_components,
    variance_threshold=0.72,
)

## 6. Silhouette Analysis

In [ ]:
plot_silhouette(
    Z=results.reduced_matrix,
    labels=results.cluster_labels,
    silhouette_samples=pipeline._clusterer.silhouette_samples_,
)

## 7. Segment Map

In [ ]:
results.segment_map()

## 8. Segment Profile Table

In [ ]:
out_df = results.to_dataframe()
profile_summary = (
    out_df.groupby(['segment_name', 'purchasing_power_tier'])
    .agg(
        n=('consumer_id', 'count'),
        adoption_lag=('adoption_lag_days', 'mean'),
        elasticity=('price_elasticity', 'mean'),
        loyalty=('loyalty_retention_ratio', 'mean'),
        attr_priority=('attribute_priority_score', 'mean'),
    )
    .round(3)
    .reset_index()
)
profile_summary

## 9. Alignment Check

In [ ]:
print(f'Alignment Score: {results.alignment_score:.3f}')
print(f'Threshold      : {pipeline.config.min_alignment_score}')
print(f'Aligned        : {results.alignment_score >= pipeline.config.min_alignment_score}')

## 10. Export

In [ ]:
results.export('caps_segment_assignments.csv')
from caps.utils.io import save_results_json
save_results_json(results, 'caps_results.json')
print('Exported: caps_segment_assignments.csv, caps_results.json')